Can independently learned semantic and trajectory classifiers provide complementary information when combined at the decision level rather than the feature level?

In [1]:
# ============================================================
# 9. LATE FUSION / STACKED GENERALIZATION
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

%load_ext autoreload
%autoreload 2

In [2]:
from scripts.taxonomy_dataset import (
    taxonomy_df,
    transformer_train_df,
    transformer_test_df,
    y_train,
    y_test,
    groups_train,
    groups_test,
    cross_dataset_splits,
)

DATASET
Total samples:       6,881
Total trajectories:  750
TRAIN / TEST SPLIT
Train samples:       5,491
Test samples:        1,390
Train trajectories:  600
Test trajectories:   150
TRAIN LABEL DISTRIBUTION
       count  percentage
label                   
-1      1457       26.53
 0       231        4.21
 1      3803       69.26
TEST LABEL DISTRIBUTION
       count  percentage
label                   
-1       402       28.92
 0        56        4.03
 1       932       67.05
LEAKAGE CHECK
Overlapping trajectories: 0
X / y / groups alignment: OK
Trajectory split:          OK
ANNOTATED FAILURES
Total: 1,859
dataset
A     179
B    1110
C     570
Name: count, dtype: int64

RULE TAXONOMY
failure_type
unknown                                640
repeated_action                        327
constraint_or_policy_violation         227
irrelevant_action                      111
missing_required_argument               62
unresolved_prior_error                  61
missing_required_action            

/Users/user/proj/agent-reliability-ml/notebooks/scripts/taxonomy_dataset.py:673: RuntimeWarning: Rule-stage unknown count: notebook checkpoint was 608, runtime reconstruction produced 640. Continuing through embedding/similarity resolution; the final training dataset is asserted exactly.
  _notebook_checkpoint(


Batches:   0%|          | 0/30 [00:00<?, ?it/s]

/Users/user/proj/agent-reliability-ml/notebooks/scripts/taxonomy_dataset.py:885: RuntimeWarning: Prototype category count: notebook checkpoint was 23, runtime reconstruction produced 20. Continuing through embedding/similarity resolution; the final training dataset is asserted exactly.
  _notebook_checkpoint(
/Users/user/proj/agent-reliability-ml/notebooks/scripts/taxonomy_dataset.py:937: RuntimeWarning: Reference-bank size: notebook checkpoint was 771, runtime reconstruction produced 758. Continuing through embedding/similarity resolution; the final training dataset is asserted exactly.
  _notebook_checkpoint(


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

...

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/Users/user/proj/agent-reliability-ml/.venv/lib/python3.12/site-packages/threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)
/Users/user/proj/agent-reliability-ml/notebooks/scripts/taxonomy_dataset.py:1262: RuntimeWarning: Remaining KMeans cl

\nNOTEBOOK REPRODUCTION CHECK
Runtime five-family samples: 1,776
Notebook reference samples: 1,779
\nRuntime family counts:
failure_family
workflow_error           798
constraint_error         387
tool_use_error           275
grounding_state_error    274
reasoning_value_error     42
Name: count, dtype: int64
\nNotebook reference family counts:
workflow_error           789
constraint_error         397
tool_use_error           274
grounding_state_error    264
reasoning_value_error     55
dtype: int64

TAXONOMY DATASET
Total samples:       1,776
Total trajectories:  419

TRAIN / TEST SPLIT
Train samples:       1,489
Test samples:        287
Train trajectories:  335
Test trajectories:   84

TRAIN LABEL DISTRIBUTION
              count  percentage
family_label                   
0               660       44.33
1               317       21.29
2               237       15.92
3               244       16.39
4                31        2.08

TEST LABEL DISTRIBUTION
              count  percentag

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Train embeddings: (1489, 384)
Train labels:     (1489,)
Test embeddings:  (287, 384)
Test labels:      (287,)


In [3]:
print("Full dataset:", taxonomy_df.shape)

print("\nTransformer split:")
print("Train:", transformer_train_df.shape)
print("Test: ", transformer_test_df.shape)

print("\nLabels:")
print("y_train:", len(y_train))
print("y_test: ", len(y_test))

print("\nGroups:")
print("Train groups:", len(set(groups_train)))
print("Test groups: ", len(set(groups_test)))

group_overlap = set(groups_train) & set(groups_test)

print("Group overlap:", len(group_overlap))

assert len(transformer_train_df) == len(y_train)
assert len(transformer_test_df) == len(y_test)

assert len(groups_train) == len(y_train)
assert len(groups_test) == len(y_test)

assert len(group_overlap) == 0

print("\n✓ Canonical group-safe split loaded correctly.")

Full dataset: (1776, 30)

Transformer split:
Train: (1489, 15)
Test:  (287, 15)

Labels:
y_train: 1489
y_test:  287

Groups:
Train groups: 335
Test groups:  84
Group overlap: 0

✓ Canonical group-safe split loaded correctly.


In [4]:
family_names = [
    "workflow_error",
    "constraint_error",
    "tool_use_error",
    "grounding_state_error",
    "reasoning_value_error",
]

id2label = {
    i: name
    for i, name in enumerate(family_names)
}

label2id = {
    name: i
    for i, name in id2label.items()
}

print("Label mapping:")
print(id2label)

print("\nTrain distribution:")
print(pd.Series(y_train).value_counts().sort_index())

print("\nTest distribution:")
print(pd.Series(y_test).value_counts().sort_index())

Label mapping:
{0: 'workflow_error', 1: 'constraint_error', 2: 'tool_use_error', 3: 'grounding_state_error', 4: 'reasoning_value_error'}

Train distribution:
family_label
0    660
1    317
2    237
3    244
4     31
Name: count, dtype: int64

Test distribution:
family_label
0    138
1     70
2     38
3     30
4     11
Name: count, dtype: int64


In [5]:
required_columns = [
    "dataset",
    "group_id",
    "current_text",
    "context_text",
    "failure_family",
    "family_label",
]

for col in required_columns:
    assert col in transformer_train_df.columns, col
    assert col in transformer_test_df.columns, col

print(
    transformer_train_df[
        [
            "dataset",
            "group_id",
            "current_text",
            "failure_family",
            "family_label",
        ]
    ].head()
)

  dataset group_id                                       current_text  \
0       A      a_1  [ASSISTANT]\nThe Australian city founded in 18...   
1       A      a_2           [ASSISTANT]\n<answer> Adelaide </answer>   
2       A      a_4  [ASSISTANT]\n<think>We need an Australian city...   
3       A      a_9  [ASSISTANT]\n<think>We need to determine wheth...   
4       A     a_29  [ASSISTANT]\nSorry — I can’t share internal ch...   

          failure_family  family_label  
0       constraint_error             1  
1       constraint_error             1  
2         tool_use_error             2  
3  grounding_state_error             3  
4  grounding_state_error             3  


In [6]:
assert (
    transformer_train_df["family_label"].to_numpy()
    == np.asarray(y_train)
).all()

assert (
    transformer_test_df["family_label"].to_numpy()
    == np.asarray(y_test)
).all()

print("✓ Text rows, labels, and groups are aligned.")

✓ Text rows, labels, and groups are aligned.


In [7]:
from sentence_transformers import SentenceTransformer

# IMPORTANT:
# Use the exact model name from Experiment 8.
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

reason_model = SentenceTransformer(MODEL_NAME)

print("Embedding dimension:",
      reason_model.get_sentence_embedding_dimension())

Embedding dimension: 384


In [8]:
train_current_texts = (
    transformer_train_df["current_text"]
    .fillna("")
    .astype(str)
    .tolist()
)

test_current_texts = (
    transformer_test_df["current_text"]
    .fillna("")
    .astype(str)
    .tolist()
)

print("Train texts:", len(train_current_texts))
print("Test texts:", len(test_current_texts))

print("\nExample:")
print(train_current_texts[0][:1000])

Train texts: 1489
Test texts: 287

Example:
[ASSISTANT]
The Australian city founded in 1838 that contains a boarding school opened by a Prime Minister of Australia and named after a school in London of the same name is Adelaide.

Reasoning:
- The search results indicate that Adelaide has a boarding school named Westminster School, which is named after Westminster School in London.
- The school was opened by Prime Minister Robert Menzies in 1961, which fits the requirement of being opened by a Prime Minister.
- Adelaide was founded in 1838, as confirmed by the historical context provided in the search results.

Thus, the city is Adelaide.

<answer>Adelaide</answer>


In [9]:
X_train_semantic = reason_model.encode(
    train_current_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

X_test_semantic = reason_model.encode(
    test_current_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

print("Semantic train:", X_train_semantic.shape)
print("Semantic test: ", X_test_semantic.shape)

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Semantic train: (1489, 384)
Semantic test:  (287, 384)


In [10]:
assert X_train_semantic.shape == (len(y_train), 384)
assert X_test_semantic.shape == (len(y_test), 384)

assert np.isfinite(X_train_semantic).all()
assert np.isfinite(X_test_semantic).all()

print("✓ Semantic representation ready.")

✓ Semantic representation ready.


In [11]:
structured_df = taxonomy_df.copy()

print(structured_df.shape)
print(structured_df.columns.tolist())

(1776, 30)
['dataset', 'group_id', 'trajectory_index', 'message_index', 'current_role', 'label', 'reason', 'context_text', 'current_text', 'is_tool_call', 'previous_messages', 'previous_tool_calls', 'previous_tool_results', 'previous_user_messages', 'previous_assistant_messages', 'context_char_length', 'context_word_count', 'current_char_length', 'current_word_count', 'reason_original', 'reason_semantic', 'taxonomy_matches', 'failure_types', 'failure_type', 'semantic_failure_type', 'semantic_similarity', 'final_failure_type', 'final_failure_type_v2', 'failure_family', 'family_label']


In [12]:
categorical_features = [
    "current_role",
    "current_tool",
    "previous_tool",
]

numeric_features = [
    "message_index",
    "is_tool_call",
    "current_char_length",
    "current_word_count",
    "context_char_length",
    "context_word_count",

    "previous_messages",
    "previous_tool_calls",
    "previous_assistant_messages",

    "parsed_tool_calls_in_context",

    "same_tool_as_previous",
    "current_tool_previous_count",
    "current_action_seen_before",

    "previous_results_have_error",
    "context_has_error_signal",

    "has_previous_tool_result",
    "has_previous_tool_call",
]

structured_features = categorical_features + numeric_features

print("Number of raw structured features:", len(structured_features))

missing = [
    col for col in structured_features
    if col not in structured_df.columns
]

print("Missing:", missing)

Number of raw structured features: 20
Missing: ['current_tool', 'previous_tool', 'parsed_tool_calls_in_context', 'same_tool_as_previous', 'current_tool_previous_count', 'current_action_seen_before', 'previous_results_have_error', 'context_has_error_signal', 'has_previous_tool_result', 'has_previous_tool_call']


In [13]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

In [16]:
import re
import pandas as pd
import numpy as np


# ============================================================
# BUILD STRUCTURED / RELATIONAL FEATURES
# ============================================================

def extract_current_tool(text):
    """
    Extract tool name from current_text.
    Returns NO_TOOL when current row is not a recognizable tool call.
    """
    text = "" if pd.isna(text) else str(text)

    # Example:
    # [TOOL_CALL]
    # search({...})
    if "[TOOL_CALL]" not in text:
        return "NO_TOOL"

    after = text.split("[TOOL_CALL]", 1)[1].strip()

    match = re.search(r"([A-Za-z_][A-Za-z0-9_]*)\s*\(", after)

    if match:
        return match.group(1)

    return "NO_TOOL"


def count_tool_calls_in_context(text):
    text = "" if pd.isna(text) else str(text)
    return text.count("[TOOL_CALL]")


def has_tool_result(text):
    text = "" if pd.isna(text) else str(text)
    return int("[TOOL_RESULT" in text)


def has_tool_call(text):
    text = "" if pd.isna(text) else str(text)
    return int("[TOOL_CALL]" in text)


ERROR_PATTERNS = [
    r"\berror\b",
    r"\bfailed\b",
    r"\bfailure\b",
    r"\bexception\b",
    r"\binvalid\b",
    r"\bnot found\b",
    r"\bdenied\b",
    r"\bunauthorized\b",
    r"\bforbidden\b",
    r"\btimeout\b",
]


def contains_error_signal(text):
    text = "" if pd.isna(text) else str(text).lower()

    return int(
        any(
            re.search(pattern, text)
            for pattern in ERROR_PATTERNS
        )
    )


# Start from canonical dataframe
structured_df = taxonomy_df.copy()


# ============================================================
# BASIC FEATURES
# ============================================================

structured_df["current_char_length"] = (
    structured_df["current_text"]
    .fillna("")
    .astype(str)
    .str.len()
)

structured_df["current_word_count"] = (
    structured_df["current_text"]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)

structured_df["context_char_length"] = (
    structured_df["context_text"]
    .fillna("")
    .astype(str)
    .str.len()
)

structured_df["context_word_count"] = (
    structured_df["context_text"]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)


# ============================================================
# CURRENT ACTION / TOOL
# ============================================================

structured_df["current_tool"] = (
    structured_df["current_text"]
    .apply(extract_current_tool)
)

structured_df["is_tool_call"] = (
    structured_df["current_role"]
    .eq("TOOL_CALL")
    .astype(int)
)


# ============================================================
# CONTEXT FEATURES
# ============================================================

structured_df["parsed_tool_calls_in_context"] = (
    structured_df["context_text"]
    .apply(count_tool_calls_in_context)
)

structured_df["has_previous_tool_result"] = (
    structured_df["context_text"]
    .apply(has_tool_result)
)

structured_df["has_previous_tool_call"] = (
    structured_df["context_text"]
    .apply(has_tool_call)
)

structured_df["context_has_error_signal"] = (
    structured_df["context_text"]
    .apply(contains_error_signal)
)


# ============================================================
# IMPORTANT:
# previous_messages / previous_tool_calls etc. are already
# COUNTS in your canonical taxonomy_df.
#
# Do NOT use len(str(value)).
# ============================================================

count_columns = [
    "previous_messages",
    "previous_tool_calls",
    "previous_tool_results",
    "previous_user_messages",
    "previous_assistant_messages",
]

for col in count_columns:
    structured_df[col] = (
        pd.to_numeric(
            structured_df[col],
            errors="coerce"
        )
        .fillna(0)
        .astype(int)
    )


# ============================================================
# PREVIOUS RESULT ERROR SIGNAL
# ============================================================
#
# taxonomy_df does not contain the raw previous tool-result
# text separately. context_text is therefore the available
# source for this proxy feature.
# ============================================================

structured_df["previous_results_have_error"] = (
    structured_df["context_text"]
    .fillna("")
    .astype(str)
    .apply(contains_error_signal)
)


# ============================================================
# RELATIONAL / TRAJECTORY FEATURES
# ============================================================

# Critical: establish trajectory order first
structured_df = (
    structured_df
    .sort_values(
        ["dataset", "group_id", "message_index"]
    )
    .reset_index(drop=True)
)


# Previous tool used in the SAME trajectory
structured_df["previous_tool"] = (
    structured_df
    .groupby(
        ["dataset", "group_id"],
        sort=False
    )["current_tool"]
    .shift(1)
    .fillna("NO_TOOL")
)


# Is current tool identical to immediately previous action?
structured_df["same_tool_as_previous"] = (
    (
        structured_df["current_tool"] ==
        structured_df["previous_tool"]
    )
    &
    (
        structured_df["current_tool"] != "NO_TOOL"
    )
).astype(int)


# ============================================================
# NUMBER OF PREVIOUS USES OF CURRENT TOOL
# ============================================================

structured_df["current_tool_previous_count"] = (
    structured_df
    .groupby(
        ["dataset", "group_id", "current_tool"],
        sort=False
    )
    .cumcount()
)

# NO_TOOL is not an actual tool/action
structured_df.loc[
    structured_df["current_tool"] == "NO_TOOL",
    "current_tool_previous_count"
] = 0


structured_df["current_action_seen_before"] = (
    structured_df["current_tool_previous_count"] > 0
).astype(int)


print("✓ Structured relational features rebuilt")
print("Shape:", structured_df.shape)

✓ Structured relational features rebuilt
Shape: (1776, 40)


In [23]:
structured_features = [
    # categorical state
    "current_role",
    "current_tool",
    "previous_tool",

    # trajectory position
    "message_index",
    "is_tool_call",

    # current/context size
    "current_char_length",
    "current_word_count",
    "context_char_length",
    "context_word_count",

    # trajectory history
    "previous_messages",
    "previous_tool_calls",
    "previous_assistant_messages",

    # visible context
    "parsed_tool_calls_in_context",
    "context_has_error_signal",
    "has_previous_tool_result",
    "has_previous_tool_call",

    # relational behavior
    "same_tool_as_previous",
    "current_tool_previous_count",
    "current_action_seen_before",
]

print("Structured features:", len(structured_features))

Structured features: 19


In [24]:
numeric_features = [
    c for c in structured_features
    if structured_df[c].dtype != "object"
]

corr = structured_df[numeric_features].corr()

high_corr = []

for i in range(len(corr.columns)):
    for j in range(i):
        value = corr.iloc[i, j]

        if abs(value) >= 0.95:
            high_corr.append({
                "feature_1": corr.columns[i],
                "feature_2": corr.columns[j],
                "correlation": value,
            })

high_corr_df = pd.DataFrame(high_corr)

high_corr_df

,feature_1,feature_2,correlation
0,current_word_count,current_char_length,0.983821
1,context_word_count,context_char_length,0.984317
2,previous_messages,message_index,0.996199
3,previous_tool_calls,message_index,0.981852
4,previous_tool_calls,previous_messages,0.993989
5,has_previous_tool_call,parsed_tool_calls_in_context,1.000000


In [26]:
key_cols = [
    "dataset",
    "group_id",
    "message_index",
]

structured_model_features = [
    "message_index",

    "current_role",
    "current_tool",
    "previous_tool",

    "is_tool_call",

    "current_char_length",
    "current_word_count",
    "context_char_length",
    "context_word_count",

    "previous_messages",
    "previous_tool_calls",
    "previous_assistant_messages",

    "parsed_tool_calls_in_context",
    "context_has_error_signal",
    "has_previous_tool_result",
    "has_previous_tool_call",

    "same_tool_as_previous",
    "current_tool_previous_count",
    "current_action_seen_before",
]

In [27]:
merge_feature_cols = list(
    dict.fromkeys(
        key_cols + structured_model_features
    )
)

print("Columns for merge:", len(merge_feature_cols))
print("Unique:", len(set(merge_feature_cols)))

assert len(merge_feature_cols) == len(set(merge_feature_cols))

Columns for merge: 21
Unique: 21


In [28]:
train_keys = transformer_train_df[
    key_cols
].copy()

test_keys = transformer_test_df[
    key_cols
].copy()


structured_train_df = train_keys.merge(
    structured_df[merge_feature_cols],
    on=key_cols,
    how="left",
    validate="one_to_one",
)

structured_test_df = test_keys.merge(
    structured_df[merge_feature_cols],
    on=key_cols,
    how="left",
    validate="one_to_one",
)

In [29]:
print(
    "Structured train:",
    structured_train_df.shape
)

print(
    "Structured test:",
    structured_test_df.shape
)

assert len(structured_train_df) == len(y_train)
assert len(structured_test_df) == len(y_test)

assert not structured_train_df[
    structured_model_features
].isna().any().any()

assert not structured_test_df[
    structured_model_features
].isna().any().any()

print("✓ Canonical alignment correct")

Structured train: (1489, 21)
Structured test: (287, 21)
✓ Canonical alignment correct


In [31]:
from sentence_transformers import SentenceTransformer

reason_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

train_texts = (
    transformer_train_df["current_text"]
    .fillna("")
    .astype(str)
    .tolist()
)

test_texts = (
    transformer_test_df["current_text"]
    .fillna("")
    .astype(str)
    .tolist()
)

X_train_current_embeddings = reason_model.encode(
    train_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

X_test_current_embeddings = reason_model.encode(
    test_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

print("Train embeddings:", X_train_current_embeddings.shape)
print("Test embeddings:", X_test_current_embeddings.shape)

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Train embeddings: (1489, 384)
Test embeddings: (287, 384)


In [32]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler


categorical_features = [
    "current_role",
    "current_tool",
    "previous_tool",
]

numeric_features = [
    c
    for c in structured_model_features
    if c not in categorical_features
]

print("Categorical:", categorical_features)
print("Numeric:", numeric_features)
print("Total:", len(categorical_features) + len(numeric_features))

Categorical: ['current_role', 'current_tool', 'previous_tool']
Numeric: ['message_index', 'is_tool_call', 'current_char_length', 'current_word_count', 'context_char_length', 'context_word_count', 'previous_messages', 'previous_tool_calls', 'previous_assistant_messages', 'parsed_tool_calls_in_context', 'context_has_error_signal', 'has_previous_tool_result', 'has_previous_tool_call', 'same_tool_as_previous', 'current_tool_previous_count', 'current_action_seen_before']
Total: 19


In [33]:
structured_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
            categorical_features,
        ),
        (
            "numeric",
            StandardScaler(),
            numeric_features,
        ),
    ],
    remainder="drop",
)


X_train_structured = structured_preprocessor.fit_transform(
    structured_train_df[structured_model_features]
)

X_test_structured = structured_preprocessor.transform(
    structured_test_df[structured_model_features]
)

print("Structured train:", X_train_structured.shape)
print("Structured test:", X_test_structured.shape)

Structured train: (1489, 222)
Structured test: (287, 222)


In [34]:
import numpy as np

X_train_fused = np.hstack([
    X_train_current_embeddings,
    X_train_structured,
])

X_test_fused = np.hstack([
    X_test_current_embeddings,
    X_test_structured,
])

print("Current embedding:", X_train_current_embeddings.shape)
print("Structured:", X_train_structured.shape)
print("Fused:", X_train_fused.shape)

assert X_train_fused.shape[0] == len(y_train)
assert X_test_fused.shape[0] == len(y_test)

print("✓ Fusion aligned")

Current embedding: (1489, 384)
Structured: (1489, 222)
Fused: (1489, 606)
✓ Fusion aligned


In [35]:
from sklearn.linear_model import LogisticRegression

fusion_classifier = LogisticRegression(
    class_weight=None,
    max_iter=5000,
    random_state=42,
)

fusion_classifier.fit(
    X_train_fused,
    y_train,
)

y_pred_fused = fusion_classifier.predict(
    X_test_fused
)

In [36]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

print("=" * 80)
print("CURRENT SEMANTIC + STRUCTURED RELATIONAL FUSION")
print("=" * 80)

print(
    classification_report(
        y_test,
        y_pred_fused,
        target_names=[
            "workflow_error",
            "constraint_error",
            "tool_use_error",
            "grounding_state_error",
            "reasoning_value_error",
        ],
        digits=4,
    )
)

accuracy = accuracy_score(
    y_test,
    y_pred_fused,
)

balanced_accuracy = balanced_accuracy_score(
    y_test,
    y_pred_fused,
)

macro_f1 = f1_score(
    y_test,
    y_pred_fused,
    average="macro",
)

weighted_f1 = f1_score(
    y_test,
    y_pred_fused,
    average="weighted",
)

print("Accuracy:", accuracy)
print("Balanced accuracy:", balanced_accuracy)
print("Macro F1:", macro_f1)
print("Weighted F1:", weighted_f1)

print("\nConfusion matrix:")
print(
    confusion_matrix(
        y_test,
        y_pred_fused,
    )
)

CURRENT SEMANTIC + STRUCTURED RELATIONAL FUSION
                       precision    recall  f1-score   support

       workflow_error     0.6048    0.5435    0.5725       138
     constraint_error     0.4706    0.4571    0.4638        70
       tool_use_error     0.3077    0.3158    0.3117        38
grounding_state_error     0.2979    0.4667    0.3636        30
reasoning_value_error     0.6667    0.5455    0.6000        11

             accuracy                         0.4843       287
            macro avg     0.4695    0.4657    0.4623       287
         weighted avg     0.5030    0.4843    0.4907       287

Accuracy: 0.4843205574912892
Balanced accuracy: 0.465706360763569
Macro F1: 0.46232237504723406
Weighted F1: 0.4906782176201221

Confusion matrix:
[[75 19 22 20  2]
 [30 32  4  4  0]
 [13  7 12  6  0]
 [ 5  9  1 14  1]
 [ 1  1  0  3  6]]


Good — this gives us a clear result, and it changes the direction of the next experiment.

### Result

Compared with your strongest `current_only` semantic baseline:

| Model / representation               |   Accuracy | Balanced Acc. |   Macro F1 | Weighted F1 |
| ------------------------------------ | ---------: | ------------: | ---------: | ----------: |
| LR current-only                      | **0.5157** |    **0.4814** | **0.4903** |  **0.5215** |
| Structured-only                      |     0.4634 |        0.4361 |     0.4476 |      0.4787 |
| Previous semantic + structured       |     0.4913 |    **0.4819** |     0.4804 |      0.4968 |
| **New semantic + relational fusion** |     0.4843 |        0.4657 |     0.4623 |      0.4907 |

So relative to current-only:

[
\Delta\text{Accuracy}=0.4843-0.5157=-0.0314
]

[
\Delta\text{Macro-F1}=0.4623-0.4903=-0.0280
]

The engineered relational features **did not improve early fusion**.

That does **not** mean the structured features contain no information. Structured-only achieved `0.4476` macro F1, which is far above a useless classifier. Rather, concatenating semantic embeddings and structured features into one feature vector does not let Logistic Regression combine the two representations beneficially.

There is also an interesting class-level effect. Fusion improves `grounding_state_error` substantially relative to your current-only LR result (`F1 ≈ 0.364` versus `0.333`), but hurts important classes such as `constraint_error`. So trajectory features appear useful for some failure mechanisms but harmful when applied uniformly.

## This is exactly why the next experiment should be late fusion / stacking

So far you did **early fusion**:

[
[\text{384 semantic dimensions};\text{structured dimensions}]
\rightarrow LR
\rightarrow y
]

The model has to learn everything simultaneously.

Instead, test:

[
Semantic\ model \rightarrow P_s(y|x)
]

and

[
Structured\ model \rightarrow P_r(y|x)
]

then combine their class probabilities:

[
[P_s;P_r] \rightarrow MetaClassifier \rightarrow y
]

This asks a different and better question:

> Can the model learn **when to trust semantic evidence and when to trust trajectory evidence?**

That's particularly relevant given your per-class results.

### Important: use OOF predictions

Do **not** train both base models on all 1,489 training rows and then predict those same 1,489 rows for the meta-classifier. That leaks training information.

We need:

```text
Training set
     │
     ├── GroupKFold
     │
     ├── semantic model ──→ OOF probabilities
     │
     └── structured model → OOF probabilities
                              │
                              ▼
                        meta classifier
                              │
Test ── base models ── probabilities ──→ meta classifier
```

And because rows belonging to the same trajectory must remain together, use your `groups_train` for the OOF folds.

### What you've learned from this stage

This sequence of experiments is becoming more informative than simply trying increasingly complicated neural networks.

You've established three different facts:

1. **Semantic current-message information is currently strongest.** `current_only` LR remains the best overall result at `0.4903` macro F1.

2. **Trajectory structure contains signal.** Structured-only reaches `0.4476` macro F1, and some classes respond differently to relational features.

3. **Signal existence ≠ successful feature fusion.** Adding informative features can make a classifier *worse*. Early concatenation dropped macro F1 from `0.4903 → 0.4623`.

That's a useful ML lesson. More features do not guarantee better generalization, especially with only **1,489 training observations**, many encoded categorical features, correlated trajectory variables, and five imbalanced classes.

So I would **not move to a custom CNN/Transformer yet**. The next controlled experiment should be:

**Experiment 9: Group-safe OOF late fusion / stacked generalization of semantic and structured classifiers.**

That will tell us whether the two representations are genuinely complementary, rather than merely individually predictive.


In [38]:
import numpy as np
from scripts.taxonomy_dataset import X_train_text_embeddings, X_test_text_embeddings

y_train = np.asarray(y_train)
y_test = np.asarray(y_test)
groups_train = np.asarray(groups_train)
groups_test = np.asarray(groups_test)

print("Semantic train:", X_train_text_embeddings.shape)
print("Semantic test: ", X_test_text_embeddings.shape)

print("Structured train:", structured_train_df.shape)
print("Structured test: ", structured_test_df.shape)

print("y:", y_train.shape, y_test.shape)
print("groups:", groups_train.shape, groups_test.shape)

assert len(X_train_text_embeddings) == len(structured_train_df) == len(y_train)
assert len(X_test_text_embeddings) == len(structured_test_df) == len(y_test)

assert set(groups_train).isdisjoint(set(groups_test))

print("✓ Alignment checks passed")

Semantic train: (1489, 384)
Semantic test:  (287, 384)
Structured train: (1489, 21)
Structured test:  (287, 21)
y: (1489,) (287,)
groups: (1489,) (287,)
✓ Alignment checks passed


In [39]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression


def make_structured_model():

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            min_frequency=2,
        )),
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ])

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=3000,
            class_weight=None,
            solver="lbfgs",
        )),
    ])

    return model

In [40]:
def make_semantic_model():

    return LogisticRegression(
        max_iter=3000,
        class_weight=None,
        solver="lbfgs",
    )

In [41]:
from sklearn.model_selection import StratifiedGroupKFold

N_SPLITS = 5

cv = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=42,
)

NUM_CLASSES = len(np.unique(y_train))

semantic_oof = np.zeros(
    (len(y_train), NUM_CLASSES),
    dtype=np.float32,
)

structured_oof = np.zeros(
    (len(y_train), NUM_CLASSES),
    dtype=np.float32,
)

In [42]:
for fold, (train_idx, val_idx) in enumerate(
    cv.split(
        X_train_text_embeddings,
        y_train,
        groups=groups_train,
    ),
    start=1,
):

    print("=" * 70)
    print(f"FOLD {fold}")
    print("=" * 70)

    # --------------------------------------------------
    # Verify group safety
    # --------------------------------------------------

    fold_train_groups = set(groups_train[train_idx])
    fold_val_groups = set(groups_train[val_idx])

    overlap = fold_train_groups & fold_val_groups

    assert len(overlap) == 0

    print("Train rows:", len(train_idx))
    print("Val rows:", len(val_idx))
    print("Group overlap:", len(overlap))

    # --------------------------------------------------
    # SEMANTIC MODEL
    # --------------------------------------------------

    semantic_model = make_semantic_model()

    semantic_model.fit(
        X_train_text_embeddings[train_idx],
        y_train[train_idx],
    )

    semantic_oof[val_idx] = semantic_model.predict_proba(
        X_train_text_embeddings[val_idx]
    )

    # --------------------------------------------------
    # STRUCTURED MODEL
    # --------------------------------------------------

    structured_model = make_structured_model()

    structured_model.fit(
        structured_train_df.iloc[train_idx],
        y_train[train_idx],
    )

    structured_oof[val_idx] = structured_model.predict_proba(
        structured_train_df.iloc[val_idx]
    )

FOLD 1
Train rows: 1190
Val rows: 299
Group overlap: 0
FOLD 2
Train rows: 1191
Val rows: 298
Group overlap: 0
FOLD 3
Train rows: 1191
Val rows: 298
Group overlap: 0
FOLD 4
Train rows: 1192
Val rows: 297
Group overlap: 0
FOLD 5
Train rows: 1192
Val rows: 297
Group overlap: 0


In [43]:
print("Semantic OOF:", semantic_oof.shape)
print("Structured OOF:", structured_oof.shape)

print("\nSemantic probability sums:")
print(semantic_oof.sum(axis=1)[:10])

print("\nStructured probability sums:")
print(structured_oof.sum(axis=1)[:10])

assert semantic_oof.shape == (len(y_train), NUM_CLASSES)
assert structured_oof.shape == (len(y_train), NUM_CLASSES)

assert np.allclose(
    semantic_oof.sum(axis=1),
    1.0,
    atol=1e-5,
)

assert np.allclose(
    structured_oof.sum(axis=1),
    1.0,
    atol=1e-5,
)

print("\n✓ OOF predictions valid")

Semantic OOF: (1489, 5)
Structured OOF: (1489, 5)

Semantic probability sums:
[0.99999994 1.         1.         0.99999994 0.99999994 1.
 1.         1.         1.         1.0000001 ]

Structured probability sums:
[1.         1.         1.         0.99999994 1.         1.
 1.         1.         1.         1.        ]

✓ OOF predictions valid


In [44]:
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)

semantic_oof_pred = semantic_oof.argmax(axis=1)
structured_oof_pred = structured_oof.argmax(axis=1)


def evaluate_predictions(name, y_true, y_pred):

    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)

    print(
        classification_report(
            y_true,
            y_pred,
            target_names=[
                id2label[i]
                for i in range(NUM_CLASSES)
            ],
            digits=4,
        )
    )

    print("Accuracy:",
          accuracy_score(y_true, y_pred))

    print("Balanced accuracy:",
          balanced_accuracy_score(y_true, y_pred))

    print("Macro F1:",
          f1_score(
              y_true,
              y_pred,
              average="macro",
          ))

    print("Weighted F1:",
          f1_score(
              y_true,
              y_pred,
              average="weighted",
          ))


evaluate_predictions(
    "OOF SEMANTIC",
    y_train,
    semantic_oof_pred,
)

evaluate_predictions(
    "OOF STRUCTURED",
    y_train,
    structured_oof_pred,
)


OOF SEMANTIC
                       precision    recall  f1-score   support

       workflow_error     0.5490    0.7303    0.6268       660
     constraint_error     0.4815    0.4101    0.4429       317
       tool_use_error     0.4380    0.2532    0.3209       237
grounding_state_error     0.5445    0.4262    0.4782       244
reasoning_value_error     0.7692    0.3226    0.4545        31

             accuracy                         0.5279      1489
            macro avg     0.5564    0.4285    0.4647      1489
         weighted avg     0.5208    0.5279    0.5110      1489

Accuracy: 0.5278710543989255
Balanced accuracy: 0.42847447556940843
Macro F1: 0.46465603575836917
Weighted F1: 0.5110100180266381

OOF STRUCTURED
                       precision    recall  f1-score   support

       workflow_error     0.5831    0.6803    0.6280       660
     constraint_error     0.4580    0.4984    0.4773       317
       tool_use_error     0.5421    0.4346    0.4824       237
grounding_state_e

In [45]:
X_meta_train = np.hstack([
    semantic_oof,
    structured_oof,
])

print(X_meta_train.shape)

(1489, 10)


In [46]:
meta_model = LogisticRegression(
    max_iter=3000,
    class_weight=None,
    solver="lbfgs",
)

meta_model.fit(
    X_meta_train,
    y_train,
)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",3000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [47]:
final_semantic_model = make_semantic_model()

final_semantic_model.fit(
    X_train_text_embeddings,
    y_train,
)


final_structured_model = make_structured_model()

final_structured_model.fit(
    structured_train_df,
    y_train,
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](5,)","[0,1,2,3,4]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](21,)","['dataset','group_id','message_index',...,'same_tool_as_previous', 'current_tool_previous_count','current_action_seen_before']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,21
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (defa

In [48]:
semantic_test_proba = final_semantic_model.predict_proba(
    X_test_text_embeddings
)

structured_test_proba = final_structured_model.predict_proba(
    structured_test_df
)

print(semantic_test_proba.shape)
print(structured_test_proba.shape)

(287, 5)
(287, 5)


In [49]:
X_meta_test = np.hstack([
    semantic_test_proba,
    structured_test_proba,
])

print("Meta train:", X_meta_train.shape)
print("Meta test:", X_meta_test.shape)

Meta train: (1489, 10)
Meta test: (287, 10)


In [50]:
stacked_pred = meta_model.predict(
    X_meta_test
)

In [51]:
from sklearn.metrics import confusion_matrix

print("\n" + "=" * 80)
print("GROUP-SAFE OOF LATE FUSION / STACKING")
print("=" * 80)

print(
    classification_report(
        y_test,
        stacked_pred,
        target_names=[
            id2label[i]
            for i in range(NUM_CLASSES)
        ],
        digits=4,
    )
)

stack_accuracy = accuracy_score(
    y_test,
    stacked_pred,
)

stack_balanced = balanced_accuracy_score(
    y_test,
    stacked_pred,
)

stack_macro = f1_score(
    y_test,
    stacked_pred,
    average="macro",
)

stack_weighted = f1_score(
    y_test,
    stacked_pred,
    average="weighted",
)

print("Accuracy:", stack_accuracy)
print("Balanced accuracy:", stack_balanced)
print("Macro F1:", stack_macro)
print("Weighted F1:", stack_weighted)

print("\nConfusion matrix:")
print(
    confusion_matrix(
        y_test,
        stacked_pred,
    )
)


GROUP-SAFE OOF LATE FUSION / STACKING
                       precision    recall  f1-score   support

       workflow_error     0.5882    0.5797    0.5839       138
     constraint_error     0.5479    0.5714    0.5594        70
       tool_use_error     0.3750    0.3158    0.3429        38
grounding_state_error     0.3171    0.4333    0.3662        30
reasoning_value_error     1.0000    0.4545    0.6250        11

             accuracy                         0.5226       287
            macro avg     0.5657    0.4710    0.4955       287
         weighted avg     0.5376    0.5226    0.5249       287

Accuracy: 0.5226480836236934
Balanced accuracy: 0.4709613955838212
Macro F1: 0.495487298247142
Weighted F1: 0.5248580755686683

Confusion matrix:
[[80 17 19 22  0]
 [28 40  1  1  0]
 [13 10 12  3  0]
 [11  6  0 13  0]
 [ 4  0  0  2  5]]


In [53]:
BEST_LR = {
    "accuracy": 0.515679,
    "balanced_accuracy": 0.481391,
    "macro_f1": 0.490268,
    "weighted_f1": 0.521487,
}

stack_results = {
    "accuracy": stack_accuracy,
    "balanced_accuracy": stack_balanced,
    "macro_f1": stack_macro,
    "weighted_f1": stack_weighted,
}

print("\n" + "=" * 80)
print("STACKING vs CURRENT-ONLY LR")
print("=" * 80)

for metric in BEST_LR:

    baseline = BEST_LR[metric]
    stacked = stack_results[metric]
    delta = stacked - baseline

    print(
        f"{metric:20s} "
        f"baseline={baseline:.4f} "
        f"stack={stacked:.4f} "
        f"delta={delta:+.4f}"
    )


STACKING vs CURRENT-ONLY LR
accuracy             baseline=0.5157 stack=0.5226 delta=+0.0070
balanced_accuracy    baseline=0.4814 stack=0.4710 delta=-0.0104
macro_f1             baseline=0.4903 stack=0.4955 delta=+0.0052
weighted_f1          baseline=0.5215 stack=0.5249 delta=+0.0034


## Experiment 9 — Group-Safe Late Fusion / Stacked Generalization

### Research objective

This experiment investigated whether **semantic information from the current message** and **structured/relational trajectory information** contain complementary signals for failure-family classification.

Previous experiments showed that directly concatenating these representations—**early fusion**—did not improve performance. Experiment 9 therefore tested whether the two information sources could be learned independently and combined at the prediction level using **late fusion / stacking**.

The five target classes were:

* `workflow_error`
* `constraint_error`
* `tool_use_error`
* `grounding_state_error`
* `reasoning_value_error`

The canonical group-safe split contained **1,489 training examples and 287 test examples**, with no trajectory-group overlap between train and test.

### Method

Two independent base classifiers were constructed.

The **semantic branch** used the 384-dimensional current-message embeddings:

[
x_{semantic}\rightarrow LogisticRegression\rightarrow P_{semantic}(y|x)
]

The **structured branch** used engineered trajectory and relational features, including message position, current and previous tool identity, previous tool activity, repeated-tool behavior, previous error signals, and related interaction features:

[
x_{structured}\rightarrow LogisticRegression\rightarrow P_{structured}(y|x)
]

To prevent leakage, training predictions for the meta-classifier were generated using **Stratified Group K-Fold out-of-fold prediction**. Consequently, every training example was predicted by models that had not been trained on its trajectory group.

Each branch produced five class probabilities. These were concatenated into a ten-dimensional meta-representation:

[
z =
[P_s(y_1),...,P_s(y_5),
P_r(y_1),...,P_r(y_5)]
]

giving:

```text
Semantic OOF:   (1489, 5)
Structured OOF: (1489, 5)
Meta training:  (1489, 10)
Meta test:      (287, 10)
```

A Logistic Regression meta-classifier was then trained on the OOF representation.

## OOF branch analysis

| Representation |   Accuracy | Balanced Accuracy |   Macro F1 | Weighted F1 |
| -------------- | ---------: | ----------------: | ---------: | ----------: |
| Semantic       |     0.5279 |            0.4285 |     0.4647 |      0.5110 |
| **Structured** | **0.5373** |        **0.4471** | **0.4810** |  **0.5293** |

One of the most important findings is that the structured trajectory representation performed strongly under group-safe OOF evaluation.

In particular:

[
F1_{structured}=0.4810
]

versus

[
F1_{semantic}=0.4647
]

This provides evidence that trajectory structure contains meaningful predictive information that is not captured solely by current-message semantics.

## Final late-fusion test result

The stacked classifier produced:

| Metric            |     Result |
| ----------------- | ---------: |
| **Accuracy**      | **0.5226** |
| Balanced Accuracy |     0.4710 |
| **Macro F1**      | **0.4955** |
| **Weighted F1**   | **0.5249** |

Per-class performance was:

| Failure family        | Precision | Recall |         F1 |
| --------------------- | --------: | -----: | ---------: |
| workflow_error        |    0.5882 | 0.5797 | **0.5839** |
| constraint_error      |    0.5479 | 0.5714 | **0.5594** |
| tool_use_error        |    0.3750 | 0.3158 |     0.3429 |
| grounding_state_error |    0.3171 | 0.4333 |     0.3662 |
| reasoning_value_error |    1.0000 | 0.4545 | **0.6250** |

The model correctly classified **150 of 287 test examples**, corresponding to 52.26% accuracy.

## Comparison with previous experiments

The most relevant comparison is between current-message semantics, early fusion, and late fusion:

| Approach                           |   Accuracy | Balanced Acc. |   Macro F1 | Weighted F1 |
| ---------------------------------- | ---------: | ------------: | ---------: | ----------: |
| Current-only LR                    |     0.5157 |    **0.4814** |     0.4903 |      0.5215 |
| Early semantic + structured fusion |     0.4843 |        0.4657 |     0.4623 |      0.4907 |
| **OOF late fusion / stacking**     | **0.5226** |        0.4710 | **0.4955** |  **0.5249** |

Relative to the strongest current-only LR baseline:

| Metric            |      Change |
| ----------------- | ----------: |
| Accuracy          | **+0.0070** |
| Balanced Accuracy | **−0.0104** |
| Macro F1          | **+0.0052** |
| Weighted F1       | **+0.0034** |

Therefore, stacking achieved the **highest observed accuracy, Macro F1, and Weighted F1** among these experiments.

### Main finding: trajectory information is useful, but fusion strategy matters

A particularly important result emerges from comparing early and late fusion.

Early fusion:

[
[semantic;structured]\rightarrow classifier
]

produced:

[
MacroF1=0.4623
]

Late fusion:

[
semantic\rightarrow P_s
]

[
structured\rightarrow P_r
]

[
[P_s;P_r]\rightarrow meta\ classifier
]

produced:

[
MacroF1=0.4955
]

This is a substantial recovery relative to early fusion:

[
0.4955-0.4623=\mathbf{+0.0332}
]

Thus, the failure of early fusion should **not** be interpreted as evidence that trajectory features are useless. Instead, the results suggest that semantic and structural representations have different statistical properties and are more effectively combined after independent modeling.

## Overall conclusion from the exploration

The experiments now support the following picture:

```text
Current message
      │
      ├── semantic representation ──────┐
      │                                 │
Trajectory/context                     ▼
      │                           class probabilities
      └── structured representation ────┤
                                        ▼
                                  Meta-classifier
                                        │
                                        ▼
                                  Failure family
```

The research has shown that **simply supplying more context does not necessarily improve failure classification**. Context-only, context+current text, DistilBERT, and early feature fusion did not outperform the simpler current-message baseline.

However, explicitly converting trajectory history into relational features revealed useful signal. The OOF results in particular demonstrate that the structured representation has meaningful group-generalizing predictive power.

Late fusion then provided a mechanism for exploiting both sources without forcing them into one raw representation.

### Research conclusion

> **Failure-family classification appears to depend on both local semantic evidence and trajectory-level structural evidence. However, these signals are not effectively combined through naive context concatenation or early feature fusion. Group-safe stacked late fusion provides modest evidence that independently modeled semantic and relational trajectory signals are complementary.**

The word **modest** is important. The best Macro-F1 improvement over current-only is only:

[
+0.0052
]

on a test set of 287 examples. Therefore, the current results establish an interesting hypothesis, but **do not yet establish a statistically reliable improvement**.

The strongest next experiment is consequently **Experiment 10 — Complementarity and Statistical Stability Analysis**: measure semantic-vs-structured disagreement and rescue cases, class-specific complementarity, oracle fusion performance, and bootstrap confidence intervals for the `+0.0052` Macro-F1 difference. That will tell you whether late fusion represents a genuine modeling improvement or ordinary test-set variation.
